In [ ]:
# ==============================================================================
# 🛰️ LEVIR CD+ Change Detection 실습 스크립트
# 📋 데이터셋 제목: LEVIR CD+ (지구 관측 기반 건물/토지 변화 탐지 데이터셋)
# 💡 의미: 이 데이터셋은 시간의 흐름에 따른 지표면 변화(예: 건물이 생기거나 사라지는 것)를
#       감지하기 위해 사용됩니다. '이미지1'과 '이미지2'가 특정 시점의 사진이라면,
#       '마스크(mask)'는 어떤 부분이 변화했는지 미리 알려주는 '정답지'입니다.
#
# 목표: 주어진 데이터를 활용하여 두 이미지 간의 '변화 에너지'를 계산하고,
#       이것이 실제 변화 마스크와 얼마나 유사한지 시각적으로 확인해보는 실습을 진행합니다.
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
import random
import os

# --- 설정 값 ---
DATASET_NAME = "blanchon/LEVIR_CDPlus"
SAMPLE_COUNT = 5 # 🔥 초보자 실습용: 전체 데이터 대신 상위 5개 샘플만 사용합니다.

print(f"✨ 튜터의 AI 한마디: 안녕하세요! 오늘 우리는 '시간을 거슬러 변화를 포착하는' 흥미진진한 작업을 해볼 거예요. 마치 지구의 시간여행자처럼요! 😉")

# ------------------------------------------------------------------------------
# 🚀 1단계: 데이터셋 로드 전략 (스트리밍 vs. 로컬)
# ------------------------------------------------------------------------------

dataset = None
split_to_use = 'train'

# 📚 가이드: 데이터셋 크기가 커서 메모리 오류가 날 수 있습니다.
# 1. 먼저 스트리밍(streaming=True)으로 시도하여 메모리 효율성을 확인합니다.
try:
    print("\n[Step 1/3] 🚀 Streaming 모드로 데이터 로드를 시도합니다...")
    # train 스플릿을 스트리밍으로 로드
    dataset = load_dataset(DATASET_NAME, split=split_to_use, streaming=True)
    print("✅ 성공! 스트리밍 모드(Streaming)로 데이터셋을 로드했습니다. 메모리 걱정 없이 탐색 가능합니다.")

except Exception as e:
    # 📉 실패할 경우: 스트리밍이 불안정할 수 있으니, 작은 데이터셋을 일반 모드로 다운로드합니다.
    print(f"\n[⚠️ 경고]: 스트리밍 로드 실패 ({e}). 일반 모드로 작은 샘플을 로드하여 진행합니다.")
    try:
        # 테스트 스플릿을 일반 모드로 제한적으로 로드하여 안정성을 확보합니다.
        dataset = load_dataset(DATASET_NAME, split='test')
        print("✅ 성공! 테스트 스플릿을 로컬 데이터셋으로 로드했습니다.")
    except Exception as e_alt:
        print(f"\n🛑 치명적인 오류: 데이터셋 로드에 실패했습니다. 오류: {e_alt}")
        exit()


# ------------------------------------------------------------------------------
# 🧪 2단계: 샘플 데이터 추출 및 전처리 (메모리 절약)
# ------------------------------------------------------------------------------

print(f"\n[Step 2/3] ✨ {SAMPLE_COUNT}개의 샘플을 추출하여 '변화 탐지'에 필요한 준비를 합니다...")

# 🛠️ 가이드: 스트리밍 모드(IterableDataset)와 일반 모드(Dataset)에 따라 샘플링 방식이 달라야 합니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)
    print("   - Streaming 모드 감지: take() 함수를 사용하여 상위 K개 샘플을 준비합니다.")
    # 스트리밍을 사용하기 위해 iterator로 변환합니다.
    sample_iterator = dataset.take(SAMPLE_COUNT)
    sample_data_list = list(sample_iterator)
else:
    # 일반 데이터셋 (Dataset)
    print("   - 일반 모드 감지: select()가 아닌 take() 패턴을 사용하여 상위 K개 샘플을 준비합니다.")
    sample_data_list = list(dataset.take(SAMPLE_COUNT))

if not sample_data_list:
    print("🚨 오류: 샘플 데이터를 전혀 가져올 수 없습니다. 로딩 설정을 확인해주세요.")
    exit()

print(f"✅ 전처리 완료: 총 {len(sample_data_list)}개의 샘플을 준비했습니다.")

# ------------------------------------------------------------------------------
# 🕵️ 3단계: 창의적인 AI 실습 - '가짜 변화 지수' 계산 및 시각화
# ------------------------------------------------------------------------------

def calculate_fake_change_index(img1, img2):
    """
    두 이미지 간의 차이를 계산하여 '가짜 변화 에너지 지수'를 만듭니다.
    (실제 AI 모델처럼 복잡하지 않지만, 수학적으로 '차이'를 계산하는 원리를 보여줍니다.)
    """
    # 🖼️ 이미지 데이터를 NumPy 배열로 변환합니다. (가장 중요!)
    img1_np = np.array(img1)
    img2_np = np.array(img2)

    # 🔍 차이 계산 (픽셀별 차이의 제곱 합)
    # 변화가 클수록(색상이 많이 다를수록) 값이 커집니다.
    difference = np.mean(np.abs(img1_np.astype(np.float32) - img2_np.astype(np.float32)), axis=2)
    
    # 차이 이미지를 0~1 사이의 값으로 정규화하여 시각화하기 쉽게 만듭니다.
    # 최대값을 이용해 차이 지수 맵을 생성합니다.
    max_diff = np.max(difference)
    if max_diff > 0:
        return difference / max_diff
    return difference

print("\n" + "="*70)
print("🖼️ [Step 3/3] 🔬 변화 탐지 실습 시작! (Image Comparison)")
print("="*70)

for i, sample in enumerate(sample_data_list):
    print(f"\n--- [샘플 {i+1}/{SAMPLE_COUNT}] 분석 중...")
    
    # 📥 데이터 추출
    try:
        # 이미지 데이터를 numpy 배열로 변환
        image1 = sample['image1']
        image2 = sample['image2']
        mask = sample['mask']
    except KeyError as e:
        print(f"🚨 오류: 필수 키가 누락되었습니다. {e}")
        continue
    
    # 💡 1. 가짜 변화 지수 계산 (Difference Map)
    change_index_map = calculate_fake_change_index(image1, image2)

    # 📊 결과를 위한 이미지 크기 확인 (예시: 1024x1024)
    try:
        # NumPy 배열로 변환 후 shape를 확인합니다.
        img1_shape = np.array(image1).shape
        img2_shape = np.array(image2).shape
        mask_shape = np.array(mask).shape
    except:
        print("⚠️ 경고: 이미지 배열 형태를 가져오는 중 오류가 발생했습니다. 건너뜁니다.")
        continue


    # 🎨 시각화 (matplotlib 사용)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # 1. T1 (Image 1) - 과거 모습
    axes[0].imshow(np.array(image1))
    axes[0].set_title("Image 1 (Time T1: Past)")
    axes[0].axis('off')
    
    # 2. T2 (Image 2) - 현재 모습
    axes[1].imshow(np.array(image2))
    axes[1].set_title("Image 2 (Time T2: Present)")
    axes[1].axis('off')
    
    # 3. Ground Truth Mask (정답)
    # 마스크는 보통 바이너리(흑백)이므로, 시각적 명확성을 위해 대비를 높여 표시합니다.
    axes[2].imshow(np.array(mask), cmap='gray')
    axes[2].set_title("Ground Truth Mask (Change)")
    axes[2].axis('off')

    # ⚡️ 가짜 변화 지수 (계산된 결과)를 보여주는 것이 가장 창의적입니다.
    plt.figure(figsize=(8, 8))
    # 'jet' 컬러맵을 사용하여 값이 클수록 (변화가 클수록) 강한 색으로 표시합니다.
    plt.imshow(change_index_map, cmap='jet') 
    plt.colorbar(label="Change Index Magnitude (Higher = More Different)")
    plt.title("Calculated Difference Map (Our Prediction)")
    plt.axis('off')
    plt.show()
    
    print(f"   ➡️ [분석 완료] Sample {i+1}: 두 이미지의 차이 지도를 생성하고, 이를 Ground Truth Mask와 시각적으로 비교했습니다.")
    print("   (변화 지수 맵의 밝은/강한 색 영역이 예상되는 변화 지역입니다.)")

print("\n" + "="*70)
print("🎉 실습 완료! 🎉")
print("축하합니다! 데이터 로드부터 분석, 그리고 가상의 AI 예측까지 모든 과정을 성공적으로 수행했습니다.")
print("✨ 기억하세요: 실제 AI에서는 이 '차이 지수'를 기반으로 딥러닝 모델을 학습시켜야 합니다.")
print("🚀 이제 데이터를 가지고 다음 단계로 나아갈 준비가 되었습니다!")
print("="*70)